# Phase 0 — AgentDojo Baseline (Colab Pro)

**Goal:** stand up AgentDojo, serve ONE small open-weight model locally with vLLM, run the **banking** suite with the **`important_instructions`** attack and **NO defense**, and print the **targeted ASR** + **utility** to check against published AgentDojo numbers.

**Success = harness trustworthy.** If ASR lands in the published ballpark, the pipeline is sound and Phase 1 can build on it. If not, fix before building anything.

**Non-negotiables honored here:**
- Deterministic success checks (AgentDojo's built-in environment-state checks) — NOT an LLM judge.
- Temperature 0 for reproducibility.
- Open-weight model served locally (no paid API).

**Runtime:** GPU required. T4 (16GB) runs a 7B in 4-bit / AWQ; L4 or A100 runs it comfortably in fp16/bf16. Set Runtime → Change runtime type → GPU.

## 0. Sanity: which GPU did Colab give us?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install AgentDojo + vLLM

vLLM gives us an OpenAI-compatible local server, which AgentDojo talks to as the `local` provider — no API key, no cost.

**numpy note:** stay on Colab's native numpy 2.x. Do NOT pin an old vllm (0.6.x) — it demands numpy<2 and collides with Colab packages compiled for numpy 2, giving the `dtype size changed` ABI error. Install modern vllm and record whatever versions resolve.

> **Before running the next cell:** Runtime → *Disconnect and delete runtime* → reconnect, so no half-downgraded numpy from an earlier attempt survives.

In [ ]:
# Modern vLLM: old 0.6.3 can't parse Qwen2.5's new rope_scaling schema
# ('assert factor in rope_scaling') and misaligns torch/transformers. A recent vllm
# understands Qwen2.5, ships a matching torch>=2.5, and keeps numpy 2 -> which also
# removes the earlier ABI error at its root (nothing downgrades numpy now).
# Let vLLM own its torch/numpy tree; agentdojo is tolerant, so install it AFTER.
!pip -q install -U vllm
!pip -q install "agentdojo==0.1.30"

# Colab's preinstalled torchaudio is built for a different CUDA than vLLM's torch (cu13 vs
# cu12); torchaudio's __init__ raises a fatal RuntimeError on the mismatch. We never use audio
# and a matching torchaudio is nightly-only -> remove it (missing = caught ImportError, no crash).
!pip -q uninstall -y torchaudio

# Record exactly what resolved — this tuple is the paper's reproducibility anchor.
import importlib.metadata as md
for p in ("vllm", "torch", "numpy", "transformers", "agentdojo"):
    try: print(p, md.version(p))
    except Exception as e: print(p, "??", e)

print("\n*** RESTART THE RUNTIME NOW: menu -> Runtime -> Restart session. ***")
print("*** Then SKIP this cell and run from step 2. ***")

## 2. Discover what THIS agentdojo version actually supports

CLI flags and provider names drift between agentdojo releases. Do not trust a remembered flag — print the truth from the installed package before running anything.

This cell dumps: the benchmark CLI help, the registered task suites (confirm `banking` exists), and the registered attacks (confirm the exact spelling of the important-instructions attack — it is usually `important_instructions`).

In [ ]:
# CLI surface — the ground truth for flags in this version.
!python -m agentdojo.scripts.benchmark --help

In [ ]:
# Registered suites and attacks. Copy the exact strings this prints into the run cell.
from agentdojo.task_suite.load_suites import get_suites
from agentdojo.attacks.attack_registry import ATTACKS

suites = get_suites("v1")
print("SUITES:", list(suites.keys()))
banking = suites["banking"]
print("banking user tasks :", len(banking.user_tasks))
print("banking inj tasks  :", len(banking.injection_tasks))
print("ATTACKS:", list(ATTACKS.keys()))

## 3. Hugging Face auth (only if the model is gated)

**Qwen2.5-7B-Instruct is NOT gated** — skip this cell and use it if the Llama license click-through isn't cleared yet (per CLAUDE.md, Qwen is the fallback).

For gated Llama-3.1-8B, paste a token (Colab Secrets → `HF_TOKEN`, or the prompt below) AND accept the license on the model's HF page first.

In [ ]:
# Only needed for gated models (e.g. Llama). Qwen needs nothing.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception as e:
    print("No Colab secret found; set os.environ['HF_TOKEN'] manually if using a gated model.", e)

## 4. Serve the model with vLLM (OpenAI-compatible, local)

Start the server in the background, then block until `/v1/models` answers. AgentDojo will hit this endpoint as the `local` provider.

**Model choice:** `Qwen/Qwen2.5-7B-Instruct` — ungated, strong tool-caller, fits the matrix. Swap `MODEL_ID` for a Llama once its license is cleared.

**T4 note:** 16GB can't hold a 7B in bf16 with KV cache for long agent traces. On T4, either use an AWQ build (`Qwen/Qwen2.5-7B-Instruct-AWQ`) or drop `--dtype` and add `--quantization awq`. On L4/A100 the fp16 path below is fine.

In [ ]:
import os, subprocess, time, urllib.request, json

# T4 has only ~14.56 GiB usable; 7B fp16 WEIGHTS alone are ~14.3 GiB -> no room for KV
# cache. Use the 4-bit AWQ build (~5.5 GiB weights) so KV cache + cudagraphs fit.
# Still the same 7B. On an A100/L4 you can switch back to the fp16 id + bfloat16.
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # ungated, 4-bit; fits a 16GB T4
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# --enable-auto-tool-choice + a parser are REQUIRED: AgentDojo drives tool calls,
# and without native tool-call parsing the agent can't act -> utility and ASR both read ~0
# for the wrong reason (broken plumbing, not a robust model). This is the #1 Phase-0 trap.
cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--port", str(PORT),
    "--quantization", "awq_marlin",   # AWQ kernels; float16 compute (T4 has no bf16)
    "--dtype", "float16",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",   # correct parser for Qwen2.5; use 'llama3_json' for Llama-3.x
]
logf = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print("vLLM starting (PID", server.pid, ") — first launch downloads weights, can take several minutes.")

In [ ]:
# Block until the server is ready (or the process dies). Tail vllm.log on failure.
def wait_ready(url, proc, timeout=1200):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            raise RuntimeError("vLLM died. Last log lines:\n" + open("vllm.log").read()[-3000:])
        try:
            with urllib.request.urlopen(url + "/models", timeout=5) as r:
                if r.status == 200:
                    print("READY:", json.loads(r.read())["data"][0]["id"])
                    return
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError("vLLM not ready in time. Check vllm.log.")

wait_ready(BASE_URL, server)

## 5. Point AgentDojo at the local server

The `local` provider reads an OpenAI-compatible base URL from the environment. The API key is a dummy — vLLM doesn't check it. The env-var name can differ by version; set both common names to be safe, and confirm against the `--help` output from step 2.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "EMPTY"   # dummy; vLLM ignores it. base_url is hardcoded to :8000.

# --- REQUIRED SHIM written to a FILE, loaded INSIDE the benchmark subprocess ---
# The benchmark runs as `python -m agentdojo.scripts.benchmark` (separate process), so an
# in-kernel monkeypatch never reaches it. agentdojo's --module-to-load imports a module at
# startup in-process; put the fix there. It flattens agentdojo's old-shape content parts
# [{'type':'text','content':...}] to a plain string, which vLLM 0.27's OpenAI server accepts
# (without it every call 400s and ASR reads a FAKE 0%). LocalLLM uses a text protocol, so
# flattening is lossless.
patch_src = '''
import agentdojo.agent_pipeline.llms.local_llm as _L
_orig = _L.chat_completion_request
def _patched(client, model, messages, **kw):
    fixed = []
    for m in messages:
        c = m.get("content")
        if isinstance(c, list):
            c = "".join((p.get("text", p.get("content", "")) if isinstance(p, dict) else str(p)) for p in c)
            m = {**m, "content": c}
        fixed.append(m)
    return _orig(client, model=model, messages=fixed, **kw)
_L.chat_completion_request = _patched
print("[patch_local] content-parts -> string applied inside benchmark process")
'''
with open("patch_local.py", "w") as f:
    f.write(patch_src)
print("Wrote patch_local.py -> load it with `-ml patch_local` in the benchmark cell.")

## 6. Run: banking suite · important_instructions · NO defense

Temperature 0, results to `./runs`. This is the whole Phase-0 experiment: one suite, one attack, no defense.

> Use the exact `--model`, `--attack`, `--defense` (omit / `none`) strings the step-2 cells printed. The flags below match agentdojo 0.1.x; adjust if `--help` disagrees.

In [ ]:
# PYTHONPATH="$PWD" makes patch_local.py importable by -ml (agentdojo's importlib didn't
# pick up cwd on its own). -ml loads the content-part shim IN this subprocess; -f ignores
# the poisoned cache. Watch for '[patch_local] ... applied' then HTTP 200 (not 400).
!ls -l patch_local.py   # must exist; if not, run the step-5 cell first
!rm -rf ./runs runs
!PYTHONPATH="$PWD" python -m agentdojo.scripts.benchmark \
  --model LOCAL \
  -s banking \
  --attack important_instructions \
  --logdir ./runs \
  -ml patch_local \
  -f

## 7. Parse results — targeted ASR + utility (deterministic)

AgentDojo writes one JSON per (user-task × injection-task) under `./runs`. Each record carries `utility` (did the intended task complete?) and `security` (did the injected attack succeed?), both from **environment-state checks**, not an LLM judge.

- **Targeted ASR** = fraction of attacked runs where the injection succeeded.
- **Utility-under-attack** = fraction where the user's real task still completed.

Field names have shifted across versions, so this parser prints the raw keys of the first record before aggregating — if `utility`/`security` aren't the keys, adjust `UTIL_KEY`/`SEC_KEY`.

In [ ]:
import json, glob

records = []
for path in glob.glob("./runs/**/*.json", recursive=True):
    try:
        with open(path) as f:
            records.append(json.load(f))
    except Exception:
        pass
assert records, "No result JSONs under ./runs — did the benchmark run to completion?"

def as_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return bool(v)
    if isinstance(v, str): return v.strip().lower() in ("true", "1", "yes")
    return False

# CRITICAL: only records with BOTH ids are real attacked runs (16 user x 9 injection = 144).
# Records with an injection_task_id but NO user_task_id are injection-FEASIBILITY rows; they
# carry security=True by construction and must be EXCLUDED or ASR is inflated (0.17 vs 0.118).
# agentdojo convention: security==True means the ATTACK SUCCEEDED (higher = worse) = ASR.
attacked = [r for r in records if r.get("user_task_id") and r.get("injection_task_id")]
n = len(attacked)
asr  = sum(as_bool(r.get("security")) for r in attacked) / n
util = sum(as_bool(r.get("utility"))  for r in attacked) / n

print(f"total records         : {len(records)}  (attacked runs used: {n})")
print(f"\n=== banking · important_instructions · no defense · Qwen2.5-7B-AWQ · T=0 ===")
print(f"targeted ASR          : {asr:.4f}   (should match agentdojo's 'Average security')")
print(f"utility-under-attack  : {util:.4f}")

## 8. The trust check

Compare the printed **targeted ASR** against the published AgentDojo `important_instructions` number **for a comparable model** (the paper reports per-model; a 7B open model is not GPT-4, so match to the closest reported open/mid model, not the top of the table).

**Verify the exact published figure against the AgentDojo paper (arXiv:2406.13352) / its results repo before declaring pass/fail** — do not anchor on a number from memory.

Interpretation:
- **ASR in the published ballpark AND utility clearly > 0** → harness trustworthy. Proceed to Phase 1 (defense matrix + adaptive attacker).
- **ASR ≈ 0 AND utility ≈ 0** → almost always broken tool-calling (wrong `--tool-call-parser`, `--enable-auto-tool-choice` missing, or the `local` provider not reaching vLLM), NOT a robust model. Fix plumbing, re-run.
- **ASR wildly higher than published** → check you're on the same suite version and that success is the env-state check, not something looser.

Record the GPU, agentdojo/vllm versions, model id, and this ASR/utility pair in the run log — that tuple is the reproducibility anchor for the paper.

In [ ]:
# Clean shutdown of the vLLM server when done (frees GPU for the next run).
try:
    server.terminate(); server.wait(timeout=30)
    print("vLLM stopped.")
except Exception as e:
    print("terminate failed, killing:", e); server.kill()